In [3]:
import os
import sys
import json

from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"

EVIDENCIAS.mkdir(parents=True, exist_ok=True)

HDFS_BASE = (
    "hdfs://localhost:9000"
    "/user/junior/datalake_afp_v2"
)

derby_path = (
    BASE / "metastore" / "metastore_db"
).as_posix()

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = (
    "/opt/hadoop-3.3.6/etc/hadoop"
)
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

spark = (
    SparkSession.builder
    .appName("AFP_V2_Calidad_Almacenamiento")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.ui.port", "4051")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        HDFS_BASE + "/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("=== CONEXIÓN SPARK ===")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(
    "Catálogo:",
    spark.conf.get("spark.sql.catalogImplementation")
)

print("\n=== BASES DE DATOS ===")

spark.sql("SHOW DATABASES").show()

=== CONEXIÓN SPARK ===
Spark: 3.5.1
Master: yarn
Catálogo: hive

=== BASES DE DATOS ===
+-------------+
|    namespace|
+-------------+
|afp_analytics|
|      default|
+-------------+



In [4]:
TABLAS = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "sbs_indicadores_adicionales",
    "tabla_analitica_mensual"
]

conteos = {}

print("=== TABLAS REGISTRADAS ===")

spark.sql("SHOW TABLES IN afp_analytics").show(
    truncate=False
)

print("=== FILAS POR TABLA ===")

for nombre in TABLAS:
    cantidad = spark.table(
        f"afp_analytics.{nombre}"
    ).count()

    conteos[nombre] = cantidad
    print(f"{nombre}: {cantidad:,}")

print(
    "\nTOTAL:",
    f"{sum(conteos.values()):,}"
)

=== TABLAS REGISTRADAS ===
+-------------+---------------------------+-----------+
|namespace    |tableName                  |isTemporary|
+-------------+---------------------------+-----------+
|afp_analytics|bcrp_json_rentabilidad     |false      |
|afp_analytics|bcrp_series_largas         |false      |
|afp_analytics|memorias_indicadores       |false      |
|afp_analytics|sbs_afiliados              |false      |
|afp_analytics|sbs_indicadores_adicionales|false      |
|afp_analytics|sbs_rentabilidad           |false      |
|afp_analytics|sbs_valor_cuota            |false      |
|afp_analytics|tabla_analitica_mensual    |false      |
+-------------+---------------------------+-----------+

=== FILAS POR TABLA ===


bcrp_series_largas: 12,317


bcrp_json_rentabilidad: 699


sbs_afiliados: 1,148


sbs_rentabilidad: 2,898


sbs_valor_cuota: 76,059


memorias_indicadores: 48


sbs_indicadores_adicionales: 17,098
tabla_analitica_mensual: 140

TOTAL: 110,407


In [5]:
CAMPOS_CRITICOS = {
    "bcrp_series_largas": [
        "codigo", "fecha_mes", "valor"
    ],
    "bcrp_json_rentabilidad": [
        "codigo", "fecha_mes", "valor"
    ],
    "sbs_afiliados": [
        "fecha_mes", "afp", "afiliados"
    ],
    "sbs_rentabilidad": [
        "fecha_mes", "afp", "fondo",
        "tipo_rentabilidad", "valor"
    ],
    "sbs_valor_cuota": [
        "fecha", "afp", "fondo",
        "valor_cuota"
    ],
    "memorias_indicadores": [
        "afp", "anio_ejercicio",
        "disponible_desde", "caracteres"
    ],
    "sbs_indicadores_adicionales": [
        "fecha", "indicador", "valor",
        "archivo_origen", "hoja_origen",
        "fila_excel", "columna_excel"
    ],
    "tabla_analitica_mensual": [
        "fecha_mes",
        "rent_real_integra_12m",
        "rent_real_habitat_12m",
        "rent_real_prima_12m",
        "rent_real_profuturo_12m",
        "ipc_lag1",
        "tipo_cambio_lag1",
        "afiliados_integra",
        "anio_ejercicio",
        "objetivo_bajo_rendimiento_mes_siguiente"
    ]
}

In [6]:
resultados_completitud = []

for nombre, campos in CAMPOS_CRITICOS.items():

    df = spark.table(f"afp_analytics.{nombre}")
    total = conteos[nombre]

    campos_disponibles = [
        c for c in campos if c in df.columns
    ]

    faltantes_esquema = [
        c for c in campos if c not in df.columns
    ]

    if faltantes_esquema:
        raise ValueError(
            f"{nombre}: columnas inexistentes: "
            f"{faltantes_esquema}"
        )

    expresiones = [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in campos_disponibles
    ]

    conteo_nulos = df.agg(*expresiones).first().asDict()

    for campo in campos_disponibles:

        nulos = int(conteo_nulos[campo] or 0)
        completos = total - nulos
        porcentaje = (
            completos / total * 100
            if total > 0 else None
        )

        resultados_completitud.append({
            "tabla": nombre,
            "campo": campo,
            "total": total,
            "completos": completos,
            "nulos": nulos,
            "completitud_pct": round(porcentaje, 2)
            if porcentaje is not None else None
        })

resumen_completitud = spark.createDataFrame(
    resultados_completitud
)

print("=== COMPLETITUD POR CAMPO CRÍTICO ===")

resumen_completitud.orderBy(
    "tabla", "campo"
).show(100, truncate=False)

=== COMPLETITUD POR CAMPO CRÍTICO ===


[Stage 53:>                                                         (0 + 1) / 2]

+---------------------------------------+---------------+---------+-----+---------------------------+-----+
|campo                                  |completitud_pct|completos|nulos|tabla                      |total|
+---------------------------------------+---------------+---------+-----+---------------------------+-----+
|codigo                                 |100.0          |699      |0    |bcrp_json_rentabilidad     |699  |
|fecha_mes                              |100.0          |699      |0    |bcrp_json_rentabilidad     |699  |
|valor                                  |100.0          |699      |0    |bcrp_json_rentabilidad     |699  |
|codigo                                 |100.0          |12317    |0    |bcrp_series_largas         |12317|
|fecha_mes                              |100.0          |12317    |0    |bcrp_series_largas         |12317|
|valor                                  |100.0          |12317    |0    |bcrp_series_largas         |12317|
|afp                        

In [7]:
CLAVES = {
    "bcrp_series_largas": [
        "codigo", "fecha_mes"
    ],
    "bcrp_json_rentabilidad": [
        "codigo", "fecha_mes"
    ],
    "sbs_afiliados": [
        "fecha_mes", "afp"
    ],
    "sbs_rentabilidad": [
        "fecha_mes", "afp",
        "fondo", "tipo_rentabilidad"
    ],
    "sbs_valor_cuota": [
        "fecha", "afp", "fondo"
    ],
    "memorias_indicadores": [
        "afp", "anio_ejercicio",
        "parte", "archivo_original"
    ],
    "sbs_indicadores_adicionales": [
        "archivo_origen", "hoja_origen",
        "fila_excel", "columna_excel"
    ],
    "tabla_analitica_mensual": [
        "fecha_mes"
    ]
}

resultados_duplicados = []

print("=== UNICIDAD DE CLAVES ===")

for nombre, claves in CLAVES.items():

    df = spark.table(f"afp_analytics.{nombre}")

    faltantes = [
        c for c in claves if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"{nombre}: claves inexistentes: {faltantes}"
        )

    repetidos = (
        df.groupBy(*claves)
        .count()
        .filter(F.col("count") > 1)
    )

    grupos = repetidos.count()

    filas_excedentes = (
        repetidos.agg(
            F.sum(F.col("count") - 1).alias("n")
        ).first()["n"] or 0
    )

    resultados_duplicados.append({
        "tabla": nombre,
        "grupos_duplicados": grupos,
        "filas_duplicadas_excedentes": int(filas_excedentes)
    })

    print(
        nombre,
        "| grupos duplicados:", grupos,
        "| filas excedentes:", int(filas_excedentes)
    )

=== UNICIDAD DE CLAVES ===


bcrp_series_largas | grupos duplicados: 0 | filas excedentes: 0


bcrp_json_rentabilidad | grupos duplicados: 0 | filas excedentes: 0


sbs_afiliados | grupos duplicados: 0 | filas excedentes: 0


sbs_rentabilidad | grupos duplicados: 0 | filas excedentes: 0


sbs_valor_cuota | grupos duplicados: 0 | filas excedentes: 0
memorias_indicadores | grupos duplicados: 0 | filas excedentes: 0


sbs_indicadores_adicionales | grupos duplicados: 0 | filas excedentes: 0
tabla_analitica_mensual | grupos duplicados: 0 | filas excedentes: 0


In [8]:
REGLAS_RANGO = [
    (
        "sbs_afiliados",
        "afiliados",
        F.col("afiliados") < 0,
        "Afiliados negativos"
    ),
    (
        "sbs_valor_cuota",
        "valor_cuota",
        F.col("valor_cuota") <= 0,
        "Valor cuota no positivo"
    ),
    (
        "memorias_indicadores",
        "caracteres",
        F.col("caracteres") <= 0,
        "Texto de memoria vacío"
    ),
    (
        "tabla_analitica_mensual",
        "tipo_cambio_lag1",
        F.col("tipo_cambio_lag1") <= 0,
        "Tipo de cambio no positivo"
    )
]

resultados_rango = []

print("=== REGLAS ESTRICTAS DE RANGO ===")

for tabla, campo, condicion, descripcion in REGLAS_RANGO:

    df = spark.table(f"afp_analytics.{tabla}")

    cantidad = df.filter(condicion).count()

    resultados_rango.append({
        "tabla": tabla,
        "campo": campo,
        "regla": descripcion,
        "incumplimientos": cantidad
    })

    print(descripcion, ":", cantidad)

=== REGLAS ESTRICTAS DE RANGO ===


Afiliados negativos : 0
Valor cuota no positivo : 0
Texto de memoria vacío : 0
Tipo de cambio no positivo : 0


In [9]:
from pyspark.sql.window import Window

final = spark.table(
    "afp_analytics.tabla_analitica_mensual"
)

print("=== COHERENCIA TEMPORAL ===")

# Memorias disponibles después del cierre del mes
memorias_futuras = final.filter(
    F.col("disponible_desde").isNotNull()
    & (
        F.col("disponible_desde")
        > F.last_day("fecha_mes")
    )
).count()

print("Memorias futuras:", memorias_futuras)

# Comprobar continuidad mensual
fechas = [
    r["fecha_mes"]
    for r in final.select("fecha_mes")
    .orderBy("fecha_mes")
    .collect()
]

from datetime import date

def mes_siguiente(fecha):
    anio = fecha.year + (fecha.month == 12)
    mes = 1 if fecha.month == 12 else fecha.month + 1
    return date(anio, mes, 1)

saltos = sum(
    mes_siguiente(a) != b
    for a, b in zip(fechas, fechas[1:])
)

print("Saltos en secuencia mensual:", saltos)

# Comparación de etiqueta contra brecha siguiente
ventana = Window.orderBy("fecha_mes")

auditoria = (
    final.select(
        "fecha_mes",
        "brecha_real_pp",
        "objetivo_bajo_rendimiento_mes_siguiente"
    )
    .withColumn(
        "brecha_siguiente",
        F.lead("brecha_real_pp").over(ventana)
    )
    .withColumn(
        "objetivo_esperado",
        F.when(
            F.col("brecha_siguiente").isNull(),
            F.lit(None).cast("int")
        ).otherwise(
            F.when(
                F.col("brecha_siguiente") < 0,
                1
            ).otherwise(0)
        )
    )
)

etiquetas_inconsistentes = auditoria.filter(
    ~F.col(
        "objetivo_bajo_rendimiento_mes_siguiente"
    ).eqNullSafe(
        F.col("objetivo_esperado")
    )
).count()

print("Etiquetas inconsistentes:", etiquetas_inconsistentes)

=== COHERENCIA TEMPORAL ===
Memorias futuras: 0
Saltos en secuencia mensual: 0
Etiquetas inconsistentes: 0


In [10]:
print("=== REGISTROS QUE INCUMPLEN REGLAS CRÍTICAS ===")

condiciones = {
    "sbs_afiliados":
        F.col("afiliados").isNull()
        | (F.col("afiliados") < 0),

    "sbs_valor_cuota":
        F.col("valor_cuota").isNull()
        | (F.col("valor_cuota") <= 0),

    "memorias_indicadores":
        F.col("caracteres").isNull()
        | (F.col("caracteres") <= 0),

    "sbs_indicadores_adicionales":
        F.col("fecha").isNull()
        | F.col("indicador").isNull()
        | F.col("valor").isNull()
        | F.col("archivo_origen").isNull()
        | F.col("hoja_origen").isNull()
        | F.col("fila_excel").isNull()
        | F.col("columna_excel").isNull()
}

incumplimientos_criticos = {}

for tabla, condicion in condiciones.items():
    df = spark.table(f"afp_analytics.{tabla}")

    cantidad = df.filter(condicion).count()

    incumplimientos_criticos[tabla] = cantidad

    print(tabla, "| filas observadas:", cantidad)

=== REGISTROS QUE INCUMPLEN REGLAS CRÍTICAS ===


sbs_afiliados | filas observadas: 0
sbs_valor_cuota | filas observadas: 0
memorias_indicadores | filas observadas: 0
sbs_indicadores_adicionales | filas observadas: 0


In [11]:
reporte_calidad = {
    "fecha": datetime.now().isoformat(),
    "filas_acumuladas": sum(conteos.values()),
    "tablas": len(TABLAS),
    "completitud": resultados_completitud,
    "duplicados": resultados_duplicados,
    "rangos": resultados_rango,
    "coherencia_temporal": {
        "memorias_futuras": memorias_futuras,
        "saltos_mensuales": saltos,
        "etiquetas_inconsistentes": etiquetas_inconsistentes
    },
    "incumplimientos_en_tablas_procesadas":
        incumplimientos_criticos,
    "nota_rechazos": (
        "No se dispone de un contador consolidado "
        "de registros rechazados durante todas las "
        "etapas históricas del ETL."
    )
}

ruta_calidad = (
    EVIDENCIAS / "16_reporte_calidad.json"
)

with ruta_calidad.open("w", encoding="utf-8") as f:
    json.dump(
        reporte_calidad,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Reporte guardado:", ruta_calidad)

Reporte guardado: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/16_reporte_calidad.json


In [12]:
print("=== FORMATO Y RUTA DE LAS TABLAS ===")

for nombre in TABLAS:

    print("\nTABLA:", nombre)

    detalle = spark.sql(
        f"DESCRIBE EXTENDED afp_analytics.{nombre}"
    )

    detalle.filter(
        F.col("col_name").isin(
            "Provider",
            "Location",
            "Partition Provider"
        )
    ).show(truncate=False)

=== FORMATO Y RUTA DE LAS TABLAS ===

TABLA: bcrp_series_largas
+------------------+------------------------------------------------------------------------------+-------+
|col_name          |data_type                                                                     |comment|
+------------------+------------------------------------------------------------------------------+-------+
|Provider          |PARQUET                                                                       |       |
|Location          |hdfs://localhost:9000/user/junior/datalake_afp_v2/processed/bcrp_series_largas|       |
|Partition Provider|Catalog                                                                       |       |
+------------------+------------------------------------------------------------------------------+-------+


TABLA: bcrp_json_rentabilidad
+------------------+----------------------------------------------------------------------------------+-------+
|col_name          |data_type       

In [16]:
print("=== VERIFICACIÓN DE PARTICIONES ===")

resultados_particiones = []

for nombre in TABLAS:

    tabla_completa = f"afp_analytics.{nombre}"

    df = spark.table(tabla_completa)

    columnas_particion = [
        c.name for c in df.schema.fields
        if c.metadata.get("isPartition", False)
    ]

    # Consultar la definición real del catálogo.
    detalles = spark.sql(
        f"DESCRIBE EXTENDED {tabla_completa}"
    ).collect()

    tiene_particiones = False

    for fila in detalles:
        if str(fila["col_name"]).strip() == "# Partition Information":
            tiene_particiones = True
            break

    print("\nTABLA:", nombre)

    if tiene_particiones:

        partes = spark.sql(
            f"SHOW PARTITIONS {tabla_completa}"
        )

        cantidad = partes.count()

        print("Estado: PARTICIONADA")
        print("Número de particiones:", cantidad)

        partes.show(5, truncate=False)

        resultados_particiones.append({
            "tabla": nombre,
            "particionada": True,
            "particiones": cantidad
        })

    else:

        print("Estado: NO PARTICIONADA")
        print("Se almacena como tabla Parquet sin particiones físicas.")

        resultados_particiones.append({
            "tabla": nombre,
            "particionada": False,
            "particiones": 0
        })

print("\n=== RESUMEN ===")

for resultado in resultados_particiones:
    print(resultado)

=== VERIFICACIÓN DE PARTICIONES ===

TABLA: bcrp_series_largas
Estado: PARTICIONADA
Número de particiones: 34
+---------+
|partition|
+---------+
|anio=1993|
|anio=1994|
|anio=1995|
|anio=1996|
|anio=1997|
+---------+
only showing top 5 rows


TABLA: bcrp_json_rentabilidad
Estado: PARTICIONADA
Número de particiones: 12
+---------+
|partition|
+---------+
|anio=2015|
|anio=2016|
|anio=2017|
|anio=2018|
|anio=2019|
+---------+
only showing top 5 rows


TABLA: sbs_afiliados
Estado: PARTICIONADA
Número de particiones: 34
+---------+
|partition|
+---------+
|anio=1993|
|anio=1994|
|anio=1995|
|anio=1996|
|anio=1997|
+---------+
only showing top 5 rows


TABLA: sbs_rentabilidad
Estado: PARTICIONADA
Número de particiones: 33
+---------+
|partition|
+---------+
|anio=1994|
|anio=1995|
|anio=1996|
|anio=1997|
|anio=1998|
+---------+
only showing top 5 rows


TABLA: sbs_valor_cuota
Estado: PARTICIONADA
Número de particiones: 34
+---------+
|partition|
+---------+
|anio=1993|
|anio=1994|
|anio=19

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 36418)
Traceback (most recent call last):
  File "/usr/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/opt/spark-3.5.1-bin-hadoop3/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/spark-3.5.1-bin-hadoop3/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/opt/spark-3.5.1-bin-hadoop3/python/pyspark/accumulators.py", line 271, in accum_updates
    n

In [1]:
import subprocess

propiedades = [
    "hadoop.security.authentication",
    "dfs.permissions.enabled",
    "dfs.encrypt.data.transfer",
    "dfs.http.policy"
]

print("=== CONFIGURACIÓN DE SEGURIDAD HADOOP ===")

for propiedad in propiedades:
    resultado = subprocess.run(
        ["hdfs", "getconf", "-confKey", propiedad],
        capture_output=True,
        text=True
    )

    if resultado.returncode == 0:
        print(f"{propiedad}: {resultado.stdout.strip()}")
    else:
        print(f"{propiedad}: NO DISPONIBLE O NO DEFINIDA")

=== CONFIGURACIÓN DE SEGURIDAD HADOOP ===
hadoop.security.authentication: simple
dfs.permissions.enabled: true
dfs.encrypt.data.transfer: false
dfs.http.policy: HTTP_ONLY
